In [ ]:
import ipywidgets as widgets
from IPython.display import display
import plotly.express as px
import pandas as pd


In [ ]:
big_df = pd.read_csv("/output/FruitCropXL/combine_all_daily_average.csv")


In [ ]:
# Define your parameters
slider_params = {
    "dayTemperature": (big_df["dayTemperature"].min(), big_df["dayTemperature"].max()),
    "nightTemperature": (big_df["nightTemperature"].min(), big_df["nightTemperature"].max()),
    "incomingRadiation": (big_df["incomingRadiation"].min(), big_df["incomingRadiation"].max()),
    "leafArea": (big_df["leafArea"].min(), big_df["leafArea"].max()),
}
dropdown_params = {
    "cca": (big_df["cca"].min(), big_df["cca"].min()),
    "totalFruitNumber": (big_df["totalFruitNumber"].min(), big_df["totalFruitNumber"].min()),
}

# Create slider widgets
slider_widgets = {
    name: widgets.FloatRangeSlider(
        value=range_, min=range_[0], max=range_[1], step=(range_[1] - range_[0]) / 100,
        description=name, continuous_update=False, layout=widgets.Layout(width='50%'), style={'description_width': 'initial'}
    )
    for name, range_ in slider_params.items()
}

# Create dropdown widgets using unique values from df
dropdown_widgets = {
    name: widgets.Dropdown(
        options=sorted(big_df[name].dropna().unique().tolist()),
        description=name,
        layout=widgets.Layout(width='200px'), style={'description_width': 'initial'}
    )
    for name in dropdown_params
}
folder_dropdown = widgets.Dropdown(
    options=["All"] + sorted(big_df["folder"].unique().tolist()),
    description="Folder:",
    layout=widgets.Layout(width='200px'), style={'description_width': 'initial'}
)

target_dropdown = widgets.Dropdown(
    options=sorted([col for col in big_df.columns if col not in slider_params and col not in dropdown_params and col != "folder"]),
    description="Target:",
    layout=widgets.Layout(width='200px'), style={'description_width': 'initial'}
)

# Filter and plot function
def update_plot(**kwargs):
    df_filtered = big_df.copy()
    
    # Filter with sliders
    for name in slider_params:
        low, high = kwargs[name]
        df_filtered = df_filtered[(df_filtered[name] >= low) & (df_filtered[name] <= high)]
    
    # Filter with dropdowns
    for name in dropdown_params:
        selected = kwargs[name]
        df_filtered = df_filtered[df_filtered[name] == selected]
    
    # Update folder dropdown options
    folder_options = ["All"] + sorted(df_filtered["folder"].unique().tolist())
    folder_dropdown.options = folder_options
    
    # Filter by folder
    folder = kwargs.get("folder_dropdown", "All")
    if folder != "All":
        df_filtered = df_filtered[df_filtered["folder"] == folder]
    
    # Plot
    target = kwargs.get("target_dropdown")
    if target and "dayOfYear" in df_filtered:
        fig = px.line(
            df_filtered, x="dayOfYear", y=target,
            color="folder",  # Different color for each folder
            title=f"{target} over Day of Year by Folder"
        )
        fig.update_traces(mode="lines+markers")
        fig.show()
    else:
        print("Missing 'dayOfYear' or target variable")

# Combine widgets
ui = widgets.VBox(
    list(slider_widgets.values()) + 
    list(dropdown_widgets.values()) + 
    [folder_dropdown, target_dropdown]
)

# Interactive binding
interactive_output = widgets.interactive_output(update_plot, {
    **slider_widgets,
    **dropdown_widgets,
    "folder_dropdown": folder_dropdown,
    "target_dropdown": target_dropdown
})

display(ui, interactive_output)


In [ ]:
big_df.head()


In [ ]:

# Example: assuming your DataFrame is called big_df
def extract_factors(folder_str):
    parts = folder_str.split('_')
    vine_type = parts[1]  # e.g., 'spur' or 'cane'
    return pd.Series([vine_type], index=['vineType'])

# Apply to DataFrame
big_df[['vineType']] = big_df['folder'].apply(extract_factors)


In [ ]:
list(big_df.columns)

In [ ]:
# after daily average, many factors is not a unique number anymore, it could be affected by the day length or others

In [ ]:
big_df['incomingRadiation'].unique()

In [ ]:
big_df['cca'].unique()

In [ ]:
big_df['Ta'].unique()

In [ ]:
big_df['leafArea'].unique()

In [ ]:
filtered_df = big_df[
    (big_df['cca'] == 900) &
    (big_df['incomingRadiation'] >= 800) &
    (
        ((big_df['vineType'] == 'spur') & (big_df['leafArea'] >= 0.2)) |
        ((big_df['vineType'] == 'cane') & (big_df['leafArea'] >= 1))
    ) & 
     (
        ((big_df['vineType'] == 'spur') & (big_df['totalFruitNumber'] <= 50)) |
        ((big_df['vineType'] == 'cane') & (big_df['totalFruitNumber'] <= 200))
    )
]


In [ ]:
# Group by vineType and nightTemperature


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Split data by vine type
spur_df = filtered_df[filtered_df['vineType'] == 'spur']
cane_df = filtered_df[filtered_df['vineType'] == 'cane']

# Create subplot with two independent y-axes
fig = make_subplots(rows=1, cols=2, subplot_titles=('Spur', 'Cane'), shared_xaxes=True)

# Spur subplot (left)
for night_temp in sorted(spur_df['Ta'].unique()):
    sub = spur_df[spur_df['Ta'] == night_temp]
    fig.add_trace(
        go.Scatter(x=sub['dayOfYear'], y=sub['biomassPlant'], mode='lines+markers',
                   name=f"NT={round(night_temp,1)}", legendgroup=f"NT={round(night_temp,1)}",
                   showlegend=(True if cane_df.empty else False)),
        row=1, col=1
    )

# Cane subplot (right)
for night_temp in sorted(cane_df['nightTemperature'].unique()):
    sub = cane_df[cane_df['nightTemperature'] == night_temp]
    fig.add_trace(
        go.Scatter(x=sub['dayOfYear'], y=sub['biomassPlant'], mode='lines+markers',
                   name=f"NT={round(night_temp,1)}", legendgroup=f"NT={round(night_temp,1)}", showlegend=True),
        row=1, col=2
    )

# Layout settings
fig.update_layout(
    title_text="Effect of Night Temperature on Biomass Dynamics (Spur vs Cane)",
    height=600,
    width=1100,
    xaxis_title="Day of Year",
    yaxis_title="Biomass (Spur)",
    yaxis2_title="Biomass (Cane)",  # This works because it's col=2
)

fig.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Split data by vine type
spur_df = filtered_df[filtered_df['vineType'] == 'spur']
cane_df = filtered_df[filtered_df['vineType'] == 'cane']

# Create subplot with two independent y-axes
fig = make_subplots(rows=1, cols=2, subplot_titles=('Spur', 'Cane'), shared_xaxes=True)

# Spur subplot (left)
for temp in sorted(spur_df['dayTemperature'].unique()):
    sub = spur_df[spur_df['dayTemperature'] == temp].sort_values('dayOfYear')
    fig.add_trace(
        go.Scatter(
            x=sub['dayOfYear'], y=sub['biomassPlant'], mode='lines+markers',
            name=f"DayT={round(temp,1)}", legendgroup=f"DayT={round(temp,1)}",
            showlegend=True if cane_df.empty else False
        ),
        row=1, col=1
    )

# Cane subplot (right)
for temp in sorted(cane_df['dayTemperature'].unique()):
    sub = cane_df[cane_df['dayTemperature'] == temp].sort_values('dayOfYear')
    fig.add_trace(
        go.Scatter(
            x=sub['dayOfYear'], y=sub['biomassPlant'], mode='lines+markers',
            name=f"DayT={round(temp,1)}", legendgroup=f"DayT={round(temp,1)}",
            showlegend=True
        ),
        row=1, col=2
    )

# Layout settings
fig.update_layout(
    title_text="Effect of Day Temperature on Biomass Dynamics (Spur vs Cane)",
    height=600,
    width=1100,
    xaxis_title="Day of Year",
    yaxis_title="Biomass (Spur)",
    yaxis2_title="Biomass (Cane)",
)

fig.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Split data by vine type
spur_df = filtered_df[filtered_df['vineType'] == 'spur']
cane_df = filtered_df[filtered_df['vineType'] == 'cane']

# Create subplot with two independent y-axes
fig = make_subplots(rows=1, cols=2, subplot_titles=('Spur', 'Cane'), shared_xaxes=True)

# Spur subplot (left)
for temp in sorted(spur_df['Ta'].unique()):
    sub = spur_df[spur_df['Ta'] == temp].sort_values('dayOfYear')
    fig.add_trace(
        go.Scatter(
            x=sub['dayOfYear'], y=sub['biomassPlant'], mode='markers',
            name=f"Ta={round(temp,1)}", legendgroup=f"Ta={round(temp,1)}",
            showlegend=True if cane_df.empty else False
        ),
        row=1, col=1
    )

# Cane subplot (right)
for temp in sorted(cane_df['Ta'].unique()):
    sub = cane_df[cane_df['Ta'] == temp].sort_values('dayOfYear')
    fig.add_trace(
        go.Scatter(
            x=sub['dayOfYear'], y=sub['biomassPlant'], mode='lines+markers',
            name=f"Ta={round(temp,1)}", legendgroup=f"Ta={round(temp,1)}",
            showlegend=True
        ),
        row=1, col=2
    )

# Layout settings
fig.update_layout(
    title_text="Effect of Air Temperature (Ta) on Biomass Dynamics (Spur vs Cane)",
    height=600,
    width=1100,
    xaxis_title="Day of Year",
    yaxis_title="Biomass (Spur)",
    yaxis2_title="Biomass (Cane)",
)

fig.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

spur_df = big_df[big_df['vineType'] == 'spur']
cane_df = big_df[big_df['vineType'] == 'cane']

# Define y-axis and group-by options
y_axis_options = ["biomassPlant", "biomassFruit", "fraction_fruitUnloading", 
                  "fraction_internodeUnloading", "phloemSugarConcentration"]
common_columns = set(spur_df.columns).intersection(cane_df.columns)
group_by_options = sorted([col for col in common_columns if col not in y_axis_options + ['dayOfYear']])

# Dropdown widgets
y_axis_dropdown = widgets.Dropdown(
    options=y_axis_options, value=y_axis_options[0], description="Y Axis:"
)
group_by_dropdown = widgets.Dropdown(
    options=group_by_options, value=group_by_options[0], description="Group By:"
)

# Update function
def update_plot(y_axis, group_by, spur_df=spur_df, cane_df=cane_df):
    fig = make_subplots(
        rows=1, cols=2, subplot_titles=('Spur', 'Cane'), shared_xaxes=True
    )

    # Add unique group ID by combining group and uuid
    spur_df = spur_df.copy()
    spur_df["group_id"] = spur_df[group_by].astype(str) + "_" + spur_df["uuid"]
    
    cane_df = cane_df.copy()
    cane_df["group_id"] = cane_df[group_by].astype(str) + "_" + cane_df["uuid"]

    # Spur subplot
    for gid in sorted(spur_df["group_id"].dropna().unique()):
        sub = spur_df[spur_df["group_id"] == gid].sort_values('dayOfYear')
        label = f"{group_by}={sub[group_by].iloc[0]}, uuid={sub['uuid'].iloc[0]}"
        fig.add_trace(
            go.Scatter(
                x=sub['dayOfYear'], y=sub[y_axis], mode='lines+markers',
                name=label, legendgroup=gid,
                showlegend=True if cane_df.empty else False
            ),
            row=1, col=1
        )

    # Cane subplot
    for gid in sorted(cane_df["group_id"].dropna().unique()):
        sub = cane_df[cane_df["group_id"] == gid].sort_values('dayOfYear')
        label = f"{group_by}={sub[group_by].iloc[0]}, uuid={sub['uuid'].iloc[0]}"
        fig.add_trace(
            go.Scatter(
                x=sub['dayOfYear'], y=sub[y_axis], mode='lines+markers',
                name=label, legendgroup=gid,
                showlegend=True
            ),
            row=1, col=2
        )

    fig.update_layout(
        title_text=f"{y_axis} vs Day of Year grouped by {group_by} (uuid-disambiguated)",
        height=600,
        width=1500,
        xaxis_title="Day of Year",
        yaxis_title=f"{y_axis} (Spur)",
        yaxis2_title=f"{y_axis} (Cane)",
    )
    fig.show()

# Interactivity
ui = widgets.VBox([y_axis_dropdown, group_by_dropdown])
out = widgets.interactive_output(update_plot, {
    'y_axis': y_axis_dropdown,
    'group_by': group_by_dropdown
})

display(ui, out)
